## Formulasi Masalah

Tujuan dari proyek ini adalah melakukan _clustering_ pada dataset *Wholesale Customers* dari UCI Machine Learning Repository untuk mengidentifikasi tiga segmen pelanggan berdasarkan pola pembelian mereka:

- **Preserved**: Konsumsi `Fresh` dan `Frozen`
- **Homers**: Konsumsi `Grocery` dan `Detergents_Paper`
- **Delight**: Konsumsi `Milk` dan `Delicassen`

Dengan melakukan segmentasi ini, perusahaan dapat menyusun strategi pemasaran yang lebih tepat sasaran.

## Eksplorasi dan Persiapan Data

Langkah-langkah:
- Memuat data
- Melakukan pembersihan (jika diperlukan)
- Melakukan normalisasi
- Melakukan visualisasi sederhana

Setiap langkah akan dijelaskan dengan alasan dan analisis hasilnya.

## Pemodelan (KMeans from Scratch)

Implementasi algoritma KMeans secara manual (tanpa library) untuk melakukan clustering terhadap data yang telah dipersiapkan.

## Evaluasi Model

Metode evaluasi yang digunakan adalah **Within-Cluster Sum of Squares (WCSS)** dan **Silhouette Score** (dihitung manual tanpa library). Evaluasi dilakukan untuk melihat seberapa baik hasil cluster yang terbentuk.

## Eksperimen Tambahan

Beberapa eksperimen dilakukan untuk membandingkan hasil:
1. Jumlah cluster berbeda (2 dan 4)
2. Normalisasi berbeda

Hasilnya dibandingkan menggunakan evaluasi dan visualisasi.

## Kesimpulan

Menjelaskan hasil akhir clustering, interpretasi tiap klaster, serta temuan utama dari eksperimen.

## Referensi

- UCI Machine Learning Repository: https://archive.ics.uci.edu/ml/datasets/Wholesale+customers
- Algoritma KMeans
- Evaluasi Clustering (WCSS, Silhouette Score)

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00292/Wholesale%20customers%20data.csv"

df = pd.read_csv(url)

df

,Channel,Region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
0,2,3,12669,9656,7561,214,2674,1338
1,2,3,7057,9810,9568,1762,3293,1776
2,2,3,6353,8808,7684,2405,3516,7844
3,1,3,13265,1196,4221,6404,507,1788
4,2,3,22615,5410,7198,3915,1777,5185
...,...,...,...,...,...,...,...,...
435,1,3,29703,12051,16027,13135,182,2204
436,1,3,39228,1431,764,4510,93,2346
437,2,3,14531,15488,30243,437,14841,1867
438,1,3,10290,1981,2232,1038,168,2125


In [ ]:
import pandas as pd
import numpy as np

def cluster_results_text(reduced_data, preds, centers, pca_samples):
    """
    Menampilkan hasil clustering dalam bentuk tabel.

    Params:
    - reduced_data: Data hasil PCA (harus 2 dimensi, kolom 'Dimension 1' dan 'Dimension 2')
    - preds: array/list hasil prediksi cluster
    - centers: array numpy dari posisi pusat cluster
    - pca_samples: array numpy dari titik sampel yang dipilih

    Return: DataFrame berisi data hasil clustering + cluster yang diprediksi
    """
    # Gabungkan data prediksi dan data PCA
    df_result = reduced_data.copy()
    df_result['Cluster'] = preds

    print("📍 Jumlah data per cluster:")
    print(df_result['Cluster'].value_counts().sort_index())
    print("\n📌 Rata-rata posisi tiap cluster (centroid berdasarkan data prediksi):")
    print(df_result.groupby('Cluster')[['Dimension 1', 'Dimension 2']].mean())

    print("\n📌 Pusat cluster dari algoritma (input `centers`):")
    for i, center in enumerate(centers):
        print(f"Cluster {i}: Dim 1 = {center[0]:.4f}, Dim 2 = {center[1]:.4f}")

    print("\n🧪 Sample data (dipilih manual):")
    for i, sample in enumerate(pca_samples):
        print(f"Sample {i}: Dim 1 = {sample[0]:.4f}, Dim 2 = {sample[1]:.4f}")

    return df_result


In [ ]:
reduced_data = pd.DataFrame({
    'Dimension 1': np.random.randn(10),
    'Dimension 2': np.random.randn(10)
})

preds = np.random.randint(0, 3, size=10)

centers = np.array([
    [0.5, -0.2],
    [-0.3, 0.9],
    [1.2, 1.1]
])

pca_samples = np.array([
    [0.3, -0.1],
    [1.0, 0.8]
])

hasil_cluster = cluster_results_text(reduced_data, preds, centers, pca_samples)

📍 Jumlah data per cluster:
Cluster
0    3
1    3
2    4
Name: count, dtype: int64

📌 Rata-rata posisi tiap cluster (centroid berdasarkan data prediksi):
         Dimension 1  Dimension 2
Cluster                          
0           0.337939    -0.753039
1           1.242574    -0.040186
2           0.593497     0.319007

📌 Pusat cluster dari algoritma (input `centers`):
Cluster 0: Dim 1 = 0.5000, Dim 2 = -0.2000
Cluster 1: Dim 1 = -0.3000, Dim 2 = 0.9000
Cluster 2: Dim 1 = 1.2000, Dim 2 = 1.1000

🧪 Sample data (dipilih manual):
Sample 0: Dim 1 = 0.3000, Dim 2 = -0.1000
Sample 1: Dim 1 = 1.0000, Dim 2 = 0.8000


# EKPLORASI DATA

In [ ]:
display(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 440 entries, 0 to 439
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Channel           440 non-null    int64
 1   Region            440 non-null    int64
 2   Fresh             440 non-null    int64
 3   Milk              440 non-null    int64
 4   Grocery           440 non-null    int64
 5   Frozen            440 non-null    int64
 6   Detergents_Paper  440 non-null    int64
 7   Delicassen        440 non-null    int64
dtypes: int64(8)
memory usage: 27.6 KB


None

In [ ]:
display(df.describe())

,Channel,Region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
count,440.000000,440.000000,440.000000,440.000000,440.000000,440.000000,440.000000,440.000000
mean,1.322727,2.543182,12000.297727,5796.265909,7951.277273,3071.931818,2881.493182,1524.870455
std,0.468052,0.774272,12647.328865,7380.377175,9503.162829,4854.673333,4767.854448,2820.105937
min,1.000000,1.000000,3.000000,55.000000,3.000000,25.000000,3.000000,3.000000
25%,1.000000,2.000000,3127.750000,1533.000000,2153.000000,742.250000,256.750000,408.250000
50%,1.000000,3.000000,8504.000000,3627.000000,4755.500000,1526.000000,816.500000,965.500000
75%,2.000000,3.000000,16933.750000,7190.250000,10655.750000,3554.250000,3922.000000,1820.250000
max,2.000000,3.000000,112151.000000,73498.000000,92780.000000,60869.000000,40827.000000,47943.000000


In [ ]:
df = pd.DataFrame({
    'Fresh': np.random.randint(1000, 20000, size=441),
    'Milk': np.random.randint(500, 10000, size=441),
    'Grocery': np.random.randint(1000, 15000, size=441),
    'Frozen': np.random.randint(100, 3000, size=441),
    'Detergents_Paper': np.random.randint(50, 4000, size=441),
    'Delicassen': np.random.randint(100, 5000, size=441)
})

# Pilih 3 indeks acak dari dataset
np.random.seed(2018)  # supaya hasil acak konsisten
indices = np.random.randint(low=0, high=len(df), size=3)
print("Indices of Samples =>", indices)

# Ambil baris data sesuai indeks tersebut
samples = pd.DataFrame(df.loc[indices], columns=df.columns).reset_index(drop=True)

# Tampilkan sampel
print("\nChosen samples of wholesale customers dataset:")
print(samples)

Indices of Samples => [250 102 226]

Chosen samples of wholesale customers dataset:
   Fresh  Milk  Grocery  Frozen  Detergents_Paper  Delicassen
0   6067  7898    13890    1421               270        1818
1   9016  3320     2013    1287              3128        3000
2   4740  5486     1007    1203               620         853


In [ ]:
data = df.copy()

# Kolom yang dipakai untuk cluster
preserved_cols = ['Fresh', 'Frozen']
homers_cols = ['Grocery', 'Detergents_Paper']
delight_cols = ['Milk', 'Delicassen']

# Hitung rata-rata populasi per kolom
population_mean = data.mean()

# Fungsi untuk print perbandingan dan insight klaster
def analyze_sample(sample):
    print("Sample data:")
    print(sample)
    print("\nSelisih terhadap rata-rata populasi:")
    diff = sample - population_mean
    print(diff)

    # Hitung total masing-masing cluster
    preserved_total = sample[preserved_cols].sum()
    preserved_mean = population_mean[preserved_cols].sum()
    homers_total = sample[homers_cols].sum()
    homers_mean = population_mean[homers_cols].sum()
    delight_total = sample[delight_cols].sum()
    delight_mean = population_mean[delight_cols].sum()

    print("\nTotal cluster (sample vs populasi):")
    print(f"Preserved (Fresh + Frozen): {preserved_total:.1f} vs {preserved_mean:.1f}")
    print(f"Homers (Grocery + Detergents_Paper): {homers_total:.1f} vs {homers_mean:.1f}")
    print(f"Delight (Milk + Delicassen): {delight_total:.1f} vs {delight_mean:.1f}")

    # Insight sederhana berdasarkan klaster
    cluster_scores = {
        'Preserved': preserved_total / preserved_mean,
        'Homers': homers_total / homers_mean,
        'Delight': delight_total / delight_mean
    }
    max_cluster = max(cluster_scores, key=cluster_scores.get)
    print(f"\nInsight: Sampel ini kemungkinan masuk klaster '{max_cluster}' karena proporsi pembelian terbesar dibanding rata-rata populasi.")

# Fungsi hitung persentil sederhana per kolom untuk sampel
def percentile_rank(sample, data):
    # Ranking (pandas rank dengan method 'average' dan pct=True)
    ranks = data.rank(pct=True)
    percentiles = {}
    for col in sample.index:
        percentiles[col] = ranks[col][data[col] <= sample[col]].max() * 100
    return percentiles

# Misal ambil 3 sampel, misalnya random seperti sebelumnya:
np.random.seed(2018)
indices = np.random.randint(low=0, high=len(data), size=3)
samples = data.iloc[indices]

# Tampilkan analisis tiap sampel
for i, sample in enumerate(samples.iterrows()):
    print(f"\n===== ANALISIS SAMPEL {i+1} (Index di data: {indices[i]}) =====")
    analyze_sample(sample[1])

    # Tampilkan persentil
    pers = percentile_rank(sample[1], data)
    print("\nPersentil pembelian tiap kategori:")
    for k,v in pers.items():
        print(f"{k}: {v:.2f}%")



===== ANALISIS SAMPEL 1 (Index di data: 250) =====
Sample data:
Fresh                6067
Milk                 7898
Grocery             13890
Frozen               1421
Detergents_Paper      270
Delicassen           1818
Name: 250, dtype: int64

Selisih terhadap rata-rata populasi:
Fresh              -4986.072562
Milk                2669.024943
Grocery             6260.333333
Frozen              -164.564626
Detergents_Paper   -1721.773243
Delicassen          -664.097506
dtype: float64

Total cluster (sample vs populasi):
Preserved (Fresh + Frozen): 7488.0 vs 12638.6
Homers (Grocery + Detergents_Paper): 14160.0 vs 9621.4
Delight (Milk + Delicassen): 9716.0 vs 7711.1

Insight: Sampel ini kemungkinan masuk klaster 'Homers' karena proporsi pembelian terbesar dibanding rata-rata populasi.

Persentil pembelian tiap kategori:
Fresh: 25.40%
Milk: 80.95%
Grocery: 93.65%
Frozen: 44.22%
Detergents_Paper: 6.80%
Delicassen: 36.28%

===== ANALISIS SAMPEL 2 (Index di data: 102) =====
Sample data:
Fre

In [ ]:
# Ambil fitur-fitur berdasarkan cluster
cluster_1 = ['Fresh', 'Frozen']
cluster_2 = ['Grocery', 'Detergents_Paper']
cluster_3 = ['Milk', 'Delicassen']

# Hitung korelasi antar fitur seluruh data
corr_matrix = df.corr()

print("=== Korelasi Fitur Keseluruhan ===")
print(corr_matrix)

# Fokus ke cluster 1
print("\n=== Korelasi Cluster 1 (Preserved: Fresh & Frozen) ===")
print(corr_matrix.loc[cluster_1, cluster_1])

# Fokus ke cluster 2
print("\n=== Korelasi Cluster 2 (Homers: Grocery & Detergents_Paper) ===")
print(corr_matrix.loc[cluster_2, cluster_2])

# Fokus ke cluster 3
print("\n=== Korelasi Cluster 3 (Delight: Milk & Delicatessen) ===")
print(corr_matrix.loc[cluster_3, cluster_3])

# Cek korelasi antar cluster (misal Fresh vs Grocery, dll)
print("\n=== Korelasi antar fitur antar cluster ===")
print(corr_matrix.loc[cluster_1, cluster_2 + cluster_3])
print(corr_matrix.loc[cluster_2, cluster_1 + cluster_3])
print(corr_matrix.loc[cluster_3, cluster_1 + cluster_2])

=== Korelasi Fitur Keseluruhan ===
                   Channel    Region     Fresh      Milk   Grocery    Frozen  \
Channel           1.000000  0.062028 -0.169172  0.460720  0.608792 -0.202046   
Region            0.062028  1.000000  0.055287  0.032288  0.007696 -0.021044   
Fresh            -0.169172  0.055287  1.000000  0.100510 -0.011854  0.345881   
Milk              0.460720  0.032288  0.100510  1.000000  0.728335  0.123994   
Grocery           0.608792  0.007696 -0.011854  0.728335  1.000000 -0.040193   
Frozen           -0.202046 -0.021044  0.345881  0.123994 -0.040193  1.000000   
Detergents_Paper  0.636026 -0.001483 -0.101953  0.661816  0.924641 -0.131525   
Delicassen        0.056011  0.045212  0.244690  0.406368  0.205497  0.390947   

                  Detergents_Paper  Delicassen  
Channel                   0.636026    0.056011  
Region                   -0.001483    0.045212  
Fresh                    -0.101953    0.244690  
Milk                      0.661816    0.406368  

In [ ]:
import pandas as pd
import numpy as np

# Data hanya berisi fitur yang dibutuhkan
features = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
data = df[features].copy()

# Fungsi menghitung R^2 score manual
def r2_score_manual(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    return 1 - ss_res / ss_tot

# Fungsi prediksi satu fitur dari yang lain
def predict_one_feature_manual(df, dropped_feature):
    print(f"\n>> Menghapus fitur '{dropped_feature}' dan memprediksi nilainya...")
    X = df.drop(columns=[dropped_feature]).values
    y = df[dropped_feature].values

    # Tambah kolom 1 di X untuk bias (intercept)
    X_bias = np.hstack([np.ones((X.shape[0], 1)), X])

    # Hitung koefisien regresi menggunakan least squares
    coeffs, _, _, _ = np.linalg.lstsq(X_bias, y, rcond=None)

    # Prediksi
    y_pred = X_bias @ coeffs

    # Hitung R^2
    score = r2_score_manual(y, y_pred)
    print(f"R² Score: {score:.3f}")
    return dropped_feature, score

# Fitur per klaster
cluster_1 = ['Fresh', 'Frozen']
cluster_2 = ['Grocery', 'Detergents_Paper']
cluster_3 = ['Milk', 'Delicassen']

print("=== Evaluasi Feature Relevance per Cluster ===")
scores = []

print("\n--- Cluster 1: Preserved ---")
for feat in cluster_1:
    scores.append(predict_one_feature_manual(data[cluster_1], feat))

print("\n--- Cluster 2: Homers ---")
for feat in cluster_2:
    scores.append(predict_one_feature_manual(data[cluster_2], feat))

print("\n--- Cluster 3: Delight ---")
for feat in cluster_3:
    scores.append(predict_one_feature_manual(data[cluster_3], feat))

# Tampilkan hasil akhir
print("\n=== Rangkuman Skor R² per Fitur ===")
for name, score in scores:
    print(f"{name:<20} : R² = {score:.3f}")


=== Evaluasi Feature Relevance per Cluster ===

--- Cluster 1: Preserved ---

>> Menghapus fitur 'Fresh' dan memprediksi nilainya...
R² Score: 0.120

>> Menghapus fitur 'Frozen' dan memprediksi nilainya...
R² Score: 0.120

--- Cluster 2: Homers ---

>> Menghapus fitur 'Grocery' dan memprediksi nilainya...
R² Score: 0.855

>> Menghapus fitur 'Detergents_Paper' dan memprediksi nilainya...
R² Score: 0.855

--- Cluster 3: Delight ---

>> Menghapus fitur 'Milk' dan memprediksi nilainya...
R² Score: 0.165

>> Menghapus fitur 'Delicassen' dan memprediksi nilainya...
R² Score: 0.165

=== Rangkuman Skor R² per Fitur ===
Fresh                : R² = 0.120
Frozen               : R² = 0.120
Grocery              : R² = 0.855
Detergents_Paper     : R² = 0.855
Milk                 : R² = 0.165
Delicassen           : R² = 0.165


In [ ]:
# Ambil hanya fitur-fitur yang digunakan
features = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
data = df[features].copy()  # Ganti df dengan nama DataFrame kamu

# Fungsi bantu untuk menampilkan distribusi dan korelasi
def analisis_cluster(cluster_data, cluster_name):
    print(f"\n=== {cluster_name} ===")

    # Statistik deskriptif
    desc = cluster_data.describe()
    print("\nStatistik Deskriptif:")
    print(desc)

    # Korelasi antar fitur
    corr = cluster_data.corr()
    print("\nKorelasi antar fitur:")
    print(corr)

    # Deteksi outlier berdasarkan IQR
    print("\nOutlier (berdasarkan IQR):")
    for col in cluster_data.columns:
        Q1 = cluster_data[col].quantile(0.25)
        Q3 = cluster_data[col].quantile(0.75)
        IQR = Q3 - Q1
        outliers = ((cluster_data[col] < (Q1 - 1.5 * IQR)) | (cluster_data[col] > (Q3 + 1.5 * IQR))).sum()
        print(f"{col}: {outliers} outlier")

    # Skewness (jika > 1 → sangat miring kanan)
    print("\nSkewness:")
    print(cluster_data.skew())

# Cluster 1: Preserved (Fresh & Frozen)
cluster1 = data[['Fresh', 'Frozen']]
analisis_cluster(cluster1, "Cluster 1 - Preserved (Fresh & Frozen)")

# Cluster 2: Homers (Grocery & Detergents_Paper)
cluster2 = data[['Grocery', 'Detergents_Paper']]
analisis_cluster(cluster2, "Cluster 2 - Homers (Grocery & Detergents_Paper)")

# Cluster 3: Delight (Milk & Delicatessen)
cluster3 = data[['Milk', 'Delicassen']]
analisis_cluster(cluster3, "Cluster 3 - Delight (Milk & Delicatessen)")



=== Cluster 1 - Preserved (Fresh & Frozen) ===

Statistik Deskriptif:
               Fresh        Frozen
count     440.000000    440.000000
mean    12000.297727   3071.931818
std     12647.328865   4854.673333
min         3.000000     25.000000
25%      3127.750000    742.250000
50%      8504.000000   1526.000000
75%     16933.750000   3554.250000
max    112151.000000  60869.000000

Korelasi antar fitur:
           Fresh    Frozen
Fresh   1.000000  0.345881
Frozen  0.345881  1.000000

Outlier (berdasarkan IQR):
Fresh: 20 outlier
Frozen: 43 outlier

Skewness:
Fresh     2.561323
Frozen    5.907986
dtype: float64

=== Cluster 2 - Homers (Grocery & Detergents_Paper) ===

Statistik Deskriptif:
            Grocery  Detergents_Paper
count    440.000000        440.000000
mean    7951.277273       2881.493182
std     9503.162829       4767.854448
min        3.000000          3.000000
25%     2153.000000        256.750000
50%     4755.500000        816.500000
75%    10655.750000       3922.0000

# Jawaban Question 3 (Versi Teks Laporan / Notebook)

**1. Distribusi Dataset:**
Berdasarkan statistik deskriptif dan nilai skewness, dapat disimpulkan bahwa *tidak ada fitur yang terdistribusi normal*. Hampir semua fitur sangat **skewed ke kanan** (nilai skewness > 2), bahkan beberapa sangat ekstrem, contohnya:

* `Frozen` (skewness ≈ 5.91)
* `Detergents_Paper` (skewness ≈ 3.63)
* `Delicatessen` (skewness ≈ 11.15)

Sebagian besar nilai berada di dekat nol (perhatikan nilai median dan Q1 yang rendah dibandingkan nilai maksimum), menunjukkan bahwa banyak customer hanya belanja sedikit untuk sebagian besar produk.

Selain itu, jumlah **outlier** juga cukup signifikan di semua fitur, seperti:

* 43 outlier di `Frozen`
* 30 outlier di `Detergents_Paper`
* 28 outlier di `Milk`
* Bahkan `Fresh` memiliki 20 outlier meskipun distribusinya lebih luas.

Hal ini menunjukkan bahwa banyak pelanggan memiliki pola belanja yang ekstrem di satu atau beberapa kategori produk.

---

**2. Korelasi antar Fitur:**
Korelasi antar fitur membantu mengidentifikasi hubungan yang kuat yang bisa berguna dalam klasterisasi. Beberapa temuan utama:

* **Cluster 2 (Homers)** menunjukkan korelasi **sangat kuat** antara:

  * `Grocery` dan `Detergents_Paper`: **0.92**
    Ini mengindikasikan bahwa pelanggan yang banyak membeli Grocery juga cenderung banyak membeli Detergents\_Paper.

* **Cluster 3 (Delight)** menampilkan korelasi **cukup** antara:

  * `Milk` dan `Delicatessen`: **0.41**
    Ini mendukung bahwa keduanya saling melengkapi dalam menggambarkan karakteristik pelanggan tertentu, walau tidak sekuat pasangan pada cluster 2.

* **Cluster 1 (Preserved)**, yang mencakup `Fresh` dan `Frozen`, hanya menunjukkan korelasi **rendah sedang** sebesar **0.34**. Ini berarti mereka relatif independen, dan bisa mewakili kebutuhan khusus seperti bahan segar/beku.

---

**3. Konfirmasi Relevansi Fitur 'Milk':**
Fitur **'Milk'** yang sebelumnya diuji untuk diprediksi (dengan R² = 0.366) ternyata:

* Tidak terlalu mudah diprediksi meskipun berkorelasi sedang dengan `Grocery` dan `Detergents_Paper`.
* Memiliki **skewness tinggi (4.05)** dan cukup banyak outlier.
* Korelasi dengan pasangannya (`Delicatessen`) juga **tidak terlalu kuat** (0.41).

Maka, keputusan untuk **mempertahankan fitur 'Milk' sebagai fitur penting tetap valid** karena perilakunya unik dan tidak sepenuhnya dapat direpresentasikan oleh fitur lain.

---

**4. Ringkasan Distribusi per Cluster:**

| Cluster   | Fitur Utama                | Skewness Tinggi                | Outlier Banyak | Korelasi Kuat |
| --------- | -------------------------- | ------------------------------ | -------------- | ------------- |
| Preserved | Fresh, Frozen              | Ya (Fresh: 2.56, Frozen: 5.91) | Ya             | Tidak terlalu |
| Homers    | Grocery, Detergents\_Paper | Ya                             | Ya             | **Ya (0.92)** |
| Delight   | Milk, Delicatessen         | **Ya (Delicatessen: 11.15)**   | Ya             | Cukup (0.41)  |


In [ ]:
import numpy as np
import pandas as pd

# Contoh data random (ganti dengan data asli kamu)
np.random.seed(42)
data = pd.DataFrame({
    'Fresh': np.random.randint(100, 1000, 100),
    'Milk': np.random.randint(50, 500, 100),
    'Grocery': np.random.randint(30, 400, 100),
    'Frozen': np.random.randint(20, 300, 100),
    'Detergents_Paper': np.random.randint(10, 200, 100),
    'Delicatessen': np.random.randint(5, 150, 100)
})

# 1. Log-transformasi data
log_data = np.log(data + 1)

# 2. Deteksi outlier dengan IQR
outliers_indices = []

for feature in log_data.columns:
    Q1 = np.percentile(log_data[feature], 25)
    Q3 = np.percentile(log_data[feature], 75)
    IQR = Q3 - Q1
    step = 1.5 * IQR

    outliers = log_data[(log_data[feature] < Q1 - step) | (log_data[feature] > Q3 + step)].index.tolist()
    outliers_indices.extend(outliers)

from collections import Counter
outliers_count = Counter(outliers_indices)
common_outliers = [idx for idx, count in outliers_count.items() if count > 1]

# 3. Buang outlier
good_data = log_data.drop(index=common_outliers).reset_index(drop=True)

# 4. PCA manual
mean_vals = good_data.mean()
data_centered = good_data - mean_vals

cov_matrix = np.cov(data_centered.T)

eig_vals, eig_vecs = np.linalg.eigh(cov_matrix)

sorted_idx = np.argsort(eig_vals)[::-1]
eig_vals = eig_vals[sorted_idx]
eig_vecs = eig_vecs[:, sorted_idx]

n_components = 2
eig_vecs_subset = eig_vecs[:, :n_components]

reduced_data = np.dot(data_centered, eig_vecs_subset)

# 5. K-Means manual
def k_means(data, k=3, max_iters=100):
    np.random.seed(42)
    centroids = data[np.random.choice(data.shape[0], k, replace=False)]

    for _ in range(max_iters):
        distances = np.linalg.norm(data[:, None] - centroids, axis=2)
        clusters = np.argmin(distances, axis=1)

        new_centroids = np.array([data[clusters == i].mean(axis=0) for i in range(k)])

        if np.allclose(new_centroids, centroids):
            break

        centroids = new_centroids

    return clusters, centroids

clusters, centroids = k_means(reduced_data, k=3)

# Tampilkan hasil clustering secara teks
print("Centroids tiap cluster (dalam ruang PCA 2D):")
for i, centroid in enumerate(centroids, 1):
    print(f"Cluster {i}: {centroid}")

print("\nJumlah data per cluster:")
unique, counts = np.unique(clusters, return_counts=True)
for u, c in zip(unique, counts):
    print(f"Cluster {u+1}: {c} data")

print("\nContoh data dari tiap cluster (index dan nilai 2D PCA):")
for i in range(3):
    print(f"\nCluster {i+1} contoh:")
    indices = np.where(clusters == i)[0][:5]  # 5 contoh
    for idx in indices:
        print(f"Index {idx}, Data: {reduced_data[idx]}")


Centroids tiap cluster (dalam ruang PCA 2D):
Cluster 1: [0.5386983  0.35276895]
Cluster 2: [-1.06663268  0.20082083]
Cluster 3: [ 0.22064475 -0.98942234]

Jumlah data per cluster:
Cluster 1: 48 data
Cluster 2: 29 data
Cluster 3: 23 data

Contoh data dari tiap cluster (index dan nilai 2D PCA):

Cluster 1 contoh:
Index 5, Data: [0.47289852 1.00766782]
Index 7, Data: [0.9919425 1.1003019]
Index 9, Data: [0.87277549 0.15689831]
Index 10, Data: [0.78750298 0.31097052]
Index 12, Data: [0.58225711 0.650901  ]

Cluster 2 contoh:
Index 0, Data: [-2.34507907  0.12814417]
Index 2, Data: [-2.03719824  0.76459383]
Index 3, Data: [-1.06060665  0.74812002]
Index 4, Data: [-0.94582076 -0.62216589]
Index 6, Data: [-2.28620507 -0.56141582]

Cluster 3 contoh:
Index 1, Data: [ 0.00265635 -1.48629826]
Index 8, Data: [ 0.28544084 -0.459281  ]
Index 15, Data: [ 0.08299221 -0.71898219]
Index 19, Data: [-0.59011079 -0.8672576 ]
Index 23, Data: [-0.36767187 -1.13619236]


In [ ]:
import numpy as np
import pandas as pd

# Contoh df sudah tersedia
# df = ... (data asli)

# 1. Log transform data, pakai log1p supaya aman kalau ada 0
log_data = np.log1p(df)

# 2. Konversi ke numpy array untuk proses clustering
data_np = log_data.values

# Fungsi euclidean distance
def euclidean_distance(a, b):
    return np.sqrt(np.sum((a - b) ** 2))

# Fungsi K-Means sederhana
def kmeans(data, k, max_iter=100, tol=1e-4, random_state=42):
    np.random.seed(random_state)
    n_samples, n_features = data.shape
    centroids = data[np.random.choice(n_samples, k, replace=False)]
    for _ in range(max_iter):
        clusters = []
        for sample in data:
            distances = [euclidean_distance(sample, centroid) for centroid in centroids]
            clusters.append(np.argmin(distances))
        clusters = np.array(clusters)
        new_centroids = np.array([data[clusters == j].mean(axis=0) for j in range(k)])
        if np.all(np.linalg.norm(new_centroids - centroids, axis=1) < tol):
            break
        centroids = new_centroids
    return clusters, centroids

# Fungsi hitung silhouette score manual
def silhouette_score_manual(data, labels):
    n_samples = data.shape[0]
    unique_labels = np.unique(labels)
    k = len(unique_labels)
    if k == 1:
        return 0
    silhouette_vals = []
    for i in range(n_samples):
        own_cluster = labels[i]
        own_cluster_points = data[labels == own_cluster]
        other_clusters = [l for l in unique_labels if l != own_cluster]

        if len(own_cluster_points) > 1:
            a = np.mean([euclidean_distance(data[i], x) for x in own_cluster_points if not np.array_equal(x, data[i])])
        else:
            a = 0

        b_values = []
        for other in other_clusters:
            other_points = data[labels == other]
            b_values.append(np.mean([euclidean_distance(data[i], x) for x in other_points]))
        b = min(b_values)

        s = (b - a) / max(a, b) if max(a, b) > 0 else 0
        silhouette_vals.append(s)

    return np.mean(silhouette_vals)

# 3. Jalankan K-Means dengan k=3
k = 3
clusters, centroids = kmeans(data_np, k)

# 4. Hitung silhouette score
score = silhouette_score_manual(data_np, clusters)
print(f'Silhouette score untuk {k} cluster: {score:.4f}')

# 5. Inverse transform centroid ke skala asli (pakai expm1 karena log1p)
true_centroids = np.expm1(centroids)

# 6. Buat DataFrame hasil centroid
centroid_df = pd.DataFrame(np.round(true_centroids), columns=df.columns)
centroid_df.index = [f'Segment {i}' for i in range(k)]
print("\nCentroid cluster (skala asli):")
print(centroid_df)

# 7. Tampilkan rata-rata data asli
print("\nRata-rata data asli:")
print(df.mean())

# 8. Contoh interpretasi singkat
for i in range(k):
    print(f"\nSegment {i} rata-rata pembelian:")
    print(centroid_df.iloc[i])


Silhouette score untuk 3 cluster: 0.1937

Centroid cluster (skala asli):
           Channel  Region    Fresh    Milk  Grocery  Frozen  \
Segment 0      1.0     2.0  14757.0  3599.0   4489.0  3957.0   
Segment 1      1.0     2.0   6115.0  1380.0   1748.0  1249.0   
Segment 2      2.0     2.0   2916.0  7913.0  12997.0   742.0   

           Detergents_Paper  Delicassen  
Segment 0             735.0      1691.0  
Segment 1             184.0       423.0  
Segment 2            5289.0       759.0  

Rata-rata data asli:
Channel                 1.322727
Region                  2.543182
Fresh               12000.297727
Milk                 5796.265909
Grocery              7951.277273
Frozen               3071.931818
Detergents_Paper     2881.493182
Delicassen           1524.870455
dtype: float64

Segment 0 rata-rata pembelian:
Channel                 1.0
Region                  2.0
Fresh               14757.0
Milk                 3599.0
Grocery              4489.0
Frozen               3957.0
D

In [ ]:
print("\nInterpretasi setiap segment berdasarkan rata-rata pembelian:")

for i in range(k):
    print(f"\nSegment {i}:")
    centroid_values = centroid_df.iloc[i]
    mean_values = df.mean()
    for col in df.columns:
        if centroid_values[col] > mean_values[col]:
            print(f"- Pembelian {col} lebih tinggi dari rata-rata")
        else:
            print(f"- Pembelian {col} lebih rendah atau sama dengan rata-rata")


Interpretasi setiap segment berdasarkan rata-rata pembelian:

Segment 0:
- Pembelian Channel lebih rendah atau sama dengan rata-rata
- Pembelian Region lebih rendah atau sama dengan rata-rata
- Pembelian Fresh lebih tinggi dari rata-rata
- Pembelian Milk lebih rendah atau sama dengan rata-rata
- Pembelian Grocery lebih rendah atau sama dengan rata-rata
- Pembelian Frozen lebih tinggi dari rata-rata
- Pembelian Detergents_Paper lebih rendah atau sama dengan rata-rata
- Pembelian Delicassen lebih tinggi dari rata-rata

Segment 1:
- Pembelian Channel lebih rendah atau sama dengan rata-rata
- Pembelian Region lebih rendah atau sama dengan rata-rata
- Pembelian Fresh lebih rendah atau sama dengan rata-rata
- Pembelian Milk lebih rendah atau sama dengan rata-rata
- Pembelian Grocery lebih rendah atau sama dengan rata-rata
- Pembelian Frozen lebih rendah atau sama dengan rata-rata
- Pembelian Detergents_Paper lebih rendah atau sama dengan rata-rata
- Pembelian Delicassen lebih rendah atau sa

In [ ]:
df['Cluster'] = clusters
print("\nContoh 5 data pertama dengan hasil cluster:")
print(df.head())


Contoh 5 data pertama dengan hasil cluster:
   Channel  Region  Fresh  Milk  Grocery  Frozen  Detergents_Paper  \
0        2       3  12669  9656     7561     214              2674   
1        2       3   7057  9810     9568    1762              3293   
2        2       3   6353  8808     7684    2405              3516   
3        1       3  13265  1196     4221    6404               507   
4        2       3  22615  5410     7198    3915              1777   

   Delicassen  Cluster  
0        1338        2  
1        1776        2  
2        7844        0  
3        1788        0  
4        5185        0  


In [ ]:
df['Cluster'] = clusters
print("\nContoh 5 data pertama dengan hasil cluster:")
df


Contoh 5 data pertama dengan hasil cluster:


,Channel,Region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen,Cluster
0,2,3,12669,9656,7561,214,2674,1338,2
1,2,3,7057,9810,9568,1762,3293,1776,2
2,2,3,6353,8808,7684,2405,3516,7844,0
3,1,3,13265,1196,4221,6404,507,1788,0
4,2,3,22615,5410,7198,3915,1777,5185,0
...,...,...,...,...,...,...,...,...,...
435,1,3,29703,12051,16027,13135,182,2204,0
436,1,3,39228,1431,764,4510,93,2346,1
437,2,3,14531,15488,30243,437,14841,1867,2
438,1,3,10290,1981,2232,1038,168,2125,1


In [ ]:
for i in range(k):
    print(f"\nSummary Segment {i}:")
    cluster_data = df[df['Cluster'] == i]
    print(f"- Jumlah data: {len(cluster_data)}")
    print(f"- Rata-rata pembelian tiap kategori:")
    print(cluster_data.drop(columns='Cluster').mean())


Summary Segment 0:
- Jumlah data: 134
- Rata-rata pembelian tiap kategori:
Channel                 1.156716
Region                  2.507463
Fresh               20331.067164
Milk                 5215.216418
Grocery              5672.985075
Frozen               6221.604478
Detergents_Paper     1103.552239
Delicassen           2687.447761
dtype: float64

Summary Segment 1:
- Jumlah data: 155
- Rata-rata pembelian tiap kategori:
Channel                 1.006452
Region                  2.535484
Fresh               10290.503226
Milk                 2017.593548
Grocery              2266.187097
Frozen               2076.206452
Detergents_Paper      321.638710
Delicassen            664.561290
dtype: float64

Summary Segment 2:
- Jumlah data: 151
- Rata-rata pembelian tiap kategori:
Channel                 1.794702
Region                  2.582781
Fresh                6362.516556
Milk                10190.668874
Grocery             15808.761589
Frozen               1298.960265
Detergents_Paper

In [ ]:
import numpy as np
import pandas as pd

# Fungsi standardisasi manual (mean=0, std=1)
def standardize(df):
    mean = df.mean()
    std = df.std()
    return (df - mean) / std

# Fungsi K-Means sederhana (tanpa sklearn)
def kmeans(X, k, max_iters=100, tol=1e-4, random_state=0):
    np.random.seed(random_state)
    n_samples, n_features = X.shape

    # Inisialisasi centroid dengan memilih k sampel acak dari data
    centroids = X[np.random.choice(n_samples, k, replace=False)]

    for _ in range(max_iters):
        # Hitung jarak euclidean dari tiap titik ke centroid
        distances = np.linalg.norm(X[:, np.newaxis] - centroids, axis=2)

        # Assign tiap titik ke cluster terdekat
        labels = np.argmin(distances, axis=1)

        # Hitung centroid baru berdasarkan rata-rata cluster
        new_centroids = np.array([X[labels == i].mean(axis=0) for i in range(k)])

        # Cek konvergensi (perubahan centroid kecil)
        if np.all(np.linalg.norm(new_centroids - centroids, axis=1) < tol):
            break

        centroids = new_centroids

    # Hitung SSE (Sum of Squared Errors)
    sse = 0
    for i in range(k):
        cluster_points = X[labels == i]
        sse += np.sum((cluster_points - centroids[i])**2)

    return labels, centroids, sse

# Contoh pakai dataframe df yang sudah berisi data numerik
# Standardisasi data
X_scaled = standardize(df)

# Hitung SSE untuk k=1 sampai 9
sse_list = []
for k in range(1, 440):
    labels, centroids, sse = kmeans(X_scaled.values, k)
    sse_list.append(sse)
    print(f"SSE untuk {k} cluster: {sse:.2f}")


SSE untuk 1 cluster: 3951.00
SSE untuk 2 cluster: 2834.45
SSE untuk 3 cluster: 2466.68
SSE untuk 4 cluster: 2086.06
SSE untuk 5 cluster: 1830.95
SSE untuk 6 cluster: 1537.52
SSE untuk 7 cluster: 1434.73
SSE untuk 8 cluster: 1366.56
SSE untuk 9 cluster: 1277.77
SSE untuk 10 cluster: 1215.20
SSE untuk 11 cluster: 1209.65
SSE untuk 12 cluster: 1187.83
SSE untuk 13 cluster: 1110.11
SSE untuk 14 cluster: 1049.76
SSE untuk 15 cluster: 1030.75
SSE untuk 16 cluster: 951.02
SSE untuk 17 cluster: 941.27
SSE untuk 18 cluster: 929.13
SSE untuk 19 cluster: 893.34
SSE untuk 20 cluster: 824.87
SSE untuk 21 cluster: 813.84
SSE untuk 22 cluster: 805.01
SSE untuk 23 cluster: 785.97
SSE untuk 24 cluster: 782.54
SSE untuk 25 cluster: 731.45
SSE untuk 26 cluster: 723.34
SSE untuk 27 cluster: 711.33
SSE untuk 28 cluster: 699.27
SSE untuk 29 cluster: 688.78
SSE untuk 30 cluster: 673.68
SSE untuk 31 cluster: 670.20
SSE untuk 32 cluster: 672.73
SSE untuk 33 cluster: 663.20
SSE untuk 34 cluster: 683.00
SSE untu

In [ ]:
kmeans = KMeans(n_clusters=439, random_state=0)
kmeans.fit(df.values)
print("SSE untuk 439 cluster:", kmeans.inertia_)

SSE untuk 439 cluster: 38897.0
